#### **FACTIBILIDAD DE LA SOLUCIÓN**

In [35]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from importlib import reload

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

Guardamos primero los tipos de cajas y productos en listas de python.

In [36]:
cajas = [
    Caja(
        caja_id = row["caja_tipo_id"],
        dim_interior_ancho = row["caja_interior_ancho"],
        dim_interior_largo = row["caja_interior_largo"],
        dim_interior_alto = row["caja_interior_alto"],
        grosor_mm = row["caja_grosor_mm"],
    )
    for _, row in especificaciones_cajas.iterrows()
]

cajas[:10]

[<Caja 016d196c89dcfcb306b853a776a879b9 | Int: 248.0x383.0x224.0mm | Ext: 257.6x392.6x233.6mm | Grosor: 4.8mm>,
 <Caja 01a2a319402ed2155292c04d8748e16f | Int: 282.0x380.0x185.0mm | Ext: 291.4x389.4x194.4mm | Grosor: 4.7mm>,
 <Caja 026560e43f3fc6afe0ce89d7ddf61626 | Int: 290.0x390.0x184.0mm | Ext: 298.2x398.2x192.2mm | Grosor: 4.1mm>,
 <Caja 02d7c6680102bd11e067c00c9b71ab9c | Int: 248.0x383.0x268.0mm | Ext: 257.6x392.6x277.6mm | Grosor: 4.8mm>,
 <Caja 0378f85c226113f4ac40fd360217bb8a | Int: 289.0x390.0x224.0mm | Ext: 297.2x398.2x232.2mm | Grosor: 4.1mm>,
 <Caja 04dce8fd28129cc6fd826678996329a4 | Int: 282.0x380.0x201.0mm | Ext: 291.4x389.4x210.4mm | Grosor: 4.7mm>,
 <Caja 057b7f8c3772129fb0b8cccf3c827c80 | Int: 290.0x390.0x286.0mm | Ext: 298.2x398.2x294.2mm | Grosor: 4.1mm>,
 <Caja 079828584749f89c6e998317886ea4cf | Int: 285.0x384.0x167.0mm | Ext: 294.4x393.4x176.4mm | Grosor: 4.7mm>,
 <Caja 093e0a1389e0cc0f644ceea470a3cc01 | Int: 284.0x379.0x289.0mm | Ext: 289.4x384.4x294.4mm | Grosor: 

In [37]:
prod_op_merge = catalogo_productos.merge(operaciones_planta, 
                                         on='codigo_producto')

productos = [
    Producto(
        codigo_producto = row['codigo_producto'],
        cantidad_paquetes = row['cantidad_paquetes'],
        peso_paquete = row['peso_neto_paquete'],
        volumen_buenos_aires = row['volumen_producto_planta_buenos_aires'],
        volumen_curitiba = row['volumen_producto_planta_curitiba'],
        volumen_santiago = row['volumen_producto_planta_santiago'],
        volumen_monterrey = row['volumen_producto_planta_monterrey'],
        volumen_bakersfield = row['volumen_producto_planta_bakersfield'],
        caja_id = row['caja_tipo_id'],
    )
    for _, row in prod_op_merge.iterrows()
]

productos[:10]

[<Producto BR0001 | Dim Prod: 285.0x386.0x303.0mm | Caja: 58c5c51d6e39e80f84be2a39cc4677a8>,
 <Producto BR0002 | Dim Prod: 290.0x390.0x260.0mm | Caja: 6ecff7742d4a01e1ada1f078a109ba73>,
 <Producto BR0003 | Dim Prod: 287.0x388.0x164.0mm | Caja: bb1778b417b37db9194a7958762985ee>,
 <Producto BR0004 | Dim Prod: 290.0x390.0x224.0mm | Caja: efbf9fb4de26ed4bd82dc7987c339263>,
 <Producto BR0005 | Dim Prod: 285.0x386.0x253.0mm | Caja: 81b0d8fe9c9138cab4b219072e8c19c3>,
 <Producto BR0006 | Dim Prod: 285.0x386.0x286.0mm | Caja: 7632b3c0a3c65e1b117ccd6cba79974b>,
 <Producto BR0007 | Dim Prod: 290.0x390.0x291.0mm | Caja: bd62436e72b572461e548cf6e4bfdd7d>,
 <Producto BR0008 | Dim Prod: 285.0x386.0x184.0mm | Caja: 18729392e748ea31ed354313be390018>,
 <Producto BR0009 | Dim Prod: 285.0x386.0x179.0mm | Caja: dbb291248e849bc6807762eb48e5f5cf>,
 <Producto BR0010 | Dim Prod: 285.0x386.0x164.0mm | Caja: bee1ee50d857c700c72bb718c6f645db>]

#### **Factibilidad 1: Tipos de cajas asignables por dimensión**

In [ ]:
productos_cajas_asignables = {}

for producto in productos:
    cajas_asignables = []
    for caja in cajas:
        if producto.es_caja_asignable(caja):
            cajas_asignables.append(caja.caja_id)
    
    productos_cajas_asignables[producto.codigo_producto] = cajas_asignables

Veamos en detalle cuántos tipos de cajas son asignables por cada producto.

In [42]:
df_productos_cajas = pd.DataFrame([
    {
        'codigo_producto': codigo,
        'cantidad_cajas_asignables': len(cajas_ids),
        'cajas_ids': cajas_ids
    }
    for codigo, cajas_ids in productos_cajas_asignables.items()
])

df_productos_cajas = df_productos_cajas.sort_values('cantidad_cajas_asignables', ascending=False)
df_productos_cajas

,codigo_producto,cantidad_cajas_asignables,cajas_ids
316,BR0317,24,"[079828584749f89c6e998317886ea4cf, 0c16d97fd1c..."
315,BR0316,24,"[079828584749f89c6e998317886ea4cf, 0c16d97fd1c..."
102,BR0103,22,"[04dce8fd28129cc6fd826678996329a4, 2672589f472..."
29,BR0030,21,"[01a2a319402ed2155292c04d8748e16f, 026560e43f3..."
283,BR0284,21,"[0378f85c226113f4ac40fd360217bb8a, 29a2bc4fc89..."
...,...,...,...
108,BR0109,1,[e6369fff40e443b558bbf80d6a5c43f6]
123,BR0124,1,[b3ea42a06319279db482c105981a393a]
344,BR0345,1,[d0823ee866ead1526b62ea77fb86dde9]
57,BR0058,1,[6d268395e153444cd7f6266651d35c77]
